In [ ]:
# 01 문제정의
# 02 데이터가져오기
# 03 EDA
# 04 데이터전처리
# 05 검증데이터 분할 및 학습
# 06 테스트 및 검증지표 확인
# 07 Model 내보내기 , 테스트파일 내보내기

In [ ]:
# IMPORT

In [1]:
import pandas as pd
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
# train = pd.read_csv('https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part2/ch2/train.csv')
# test = pd.read_csv('https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part2/ch2/test.csv')

print('train:', train.shape, '| test:', test.shape)

train: (29304, 16) | test: (3257, 15)


In [2]:
# y값 분리
y = train['income'].map({'<=50K': 0, '>50K': 1})   # 문자 → 0/1

In [3]:
train=train.select_dtypes(include="number");
test=test.select_dtypes(include="number");
train=train.drop(columns=['fnlwgt','capital.gain','capital.loss'])
test=test.drop(columns=['fnlwgt','capital.gain','capital.loss'])

In [ ]:
# 데이터 전처리

In [4]:
num_cols = ['age', 'hours.per.week']  # 수치형 데이터

for c in num_cols:
    m = train[c].median()            # train 기준 중앙값
    train[c] = train[c].fillna(m)
    test[c] = test[c].fillna(m)

In [5]:
train = train.drop(columns=['id'])

In [6]:
test_id = test['id']                                # 제출용 보관
test = test.drop(columns=['id'])

In [7]:
all_df = pd.concat([train, test], axis=0)           # 합쳐서 인코딩
all_oh = pd.get_dummies(all_df)

In [8]:
X = all_oh.iloc[:len(train)]                        # 다시 train 부분
X_test = all_oh.iloc[len(train):]                   # test 부분
print('인코딩 후 컬럼 수:', X.shape[1])

인코딩 후 컬럼 수: 3


In [ ]:
# 검증 데이터 분할

In [9]:
from sklearn.model_selection import train_test_split
X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y)
print(X_tr.shape, X_val.shape)

(23443, 3) (5861, 3)


In [ ]:
# 데이터 학습

In [10]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=100, random_state=0)
rf.fit(X_tr, y_tr)

RandomForestClassifier(random_state=0)

In [ ]:
# 모델 검증 지표

In [11]:
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
pred = rf.predict(X_val)
print(pred)
proba = rf.predict_proba(X_val)[:, 1]   # 1(>50K)일 확률 열만 (2차원 슬라이싱)
print(proba)
print('정확도 :', round(accuracy_score(y_val, pred), 4))
print('ROC-AUC:', round(roc_auc_score(y_val, proba), 4))
# # print(classification_report(y_val, pred))

[0 0 1 ... 0 0 0]
[0.47625017 0.01133846 0.7475751  ... 0.         0.         0.        ]
정확도 : 0.7674
ROC-AUC: 0.764


In [ ]:
# model.pkl 파일변환

In [18]:

# model.pkl 파일변환(model 만묶기)
import joblib

joblib.dump(rf, "model.pkl")             # 2) 파일로 저장

['model.pkl']

In [12]:
import joblib
# 모델 feature , target 묶기
bundle = {
    "model": rf,
    "features": list(X_tr.columns),          # 학습에 쓴 컬럼 순서
    "targets": ["<=50K", ">50K"],               # 클래스 라벨
}
joblib.dump(bundle, "model.pkl")

['model.pkl']